
# Laboratorio: Ajustando la Capacidad de la Red (Fashion-MNIST)

En este ejercicio aprenderemos a identificar los dos grandes enemigos del Deep Learning: el **Underfitting** y el **Overfitting**, ajustando manualmente el número de capas y neuronas.

## El Dataset: Fashion-MNIST
A diferencia de CIFAR-10 (fotos a color), Fashion-MNIST son imágenes en **escala de grises** (un solo canal) de 28x28 píxeles.
Contiene 10 categorías: T-shirt, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle boot.


In [1]:
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import numpy as np

# Cargar datos
fashion_mnist = keras.datasets.fashion_mnist
(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()

# Normalización (Escalar de 0-255 a 0-1)
train_images = train_images / 255.0
test_images = test_images / 255.0

# Añadir una dimensión extra para el canal (necesario para Conv2D en escala de grises)
train_images = train_images.reshape((train_images.shape[0], 28, 28, 1))
test_images = test_images.reshape((test_images.shape[0], 28, 28, 1))

print(f"Imágenes de entrenamiento: {train_images.shape}")

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Imágenes de entrenamiento: (60000, 28, 28, 1)



El **Underfitting** ocurre cuando el modelo es demasiado simple. Es como intentar aprender física cuántica usando solo sumas y restas; el modelo no tiene "capacidad" suficiente para entender los patrones.

**Tu Tarea:** Ejecuta el siguiente modelo "minimalista" (solo una capa densa muy pequeña) y observa el accuracy.



In [2]:
# Modelo ultra-simple
model_under = keras.Sequential([
    keras.layers.Flatten(input_shape=(28, 28, 1)),
    keras.layers.Dense(4, activation='relu'), # ¡Solo 4 neuronas!
    keras.layers.Dense(10, activation='softmax')
])

model_under.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history_under = model_under.fit(train_images, train_labels, epochs=10,
                                validation_data=(test_images, test_labels), verbose=0)

print(f"Precisión Entrenamiento: {history_under.history['accuracy'][-1]:.4f}")
print(f"Precisión Test: {history_under.history['val_accuracy'][-1]:.4f}")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Precisión Entrenamiento: 0.8256
Precisión Test: 0.8147



## Experimento 2: Identificando el Overfitting (Sobreajuste)

El **Overfitting** ocurre cuando el modelo es demasiado complejo y tiene demasiadas capas o neuronas para un problema sencillo. En lugar de aprender a reconocer una "bota", **memoriza** las imágenes de entrenamiento.

**Tu Tarea:** Entrena este modelo masivo durante muchas épocas y observa cómo el error de entrenamiento sigue bajando, pero el de test se estanca o sube.



In [3]:
# Modelo exageradamente grande para este problema
model_over = keras.Sequential([
    keras.layers.Conv2D(128, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    keras.layers.Conv2D(128, (3, 3), activation='relu'),
    keras.layers.Flatten(),
    keras.layers.Dense(512, activation='relu'),
    keras.layers.Dense(512, activation='relu'),
    keras.layers.Dense(10, activation='softmax')
])

model_over.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Entrenamos por 20 épocas (puede tardar un poco)
history_over = model_over.fit(train_images, train_labels, epochs=20,
                              validation_data=(test_images, test_labels), verbose=1)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/20
 542/1875 ━━━━━━━━━━━━━━━━━━━━ 13:33 611ms/step - accuracy: 0.7649 - loss: 0.6589

KeyboardInterrupt: 


## Tarea Final: Encontrar el "Punto Dulce"

Basándote en los dos experimentos anteriores, tu misión es crear un modelo equilibrado.

### Instrucciones para los alumnos:
1.  **Si hay Underfitting (Error alto en ambos):** Debes **aumentar** la capacidad (añadir más filtros Conv2D o más neuronas en Dense).
2.  **Si hay Overfitting (Entrenamiento perfecto, Test malo):** Debes **reducir** capas o añadir técnicas de regularización como "Dropout".

**Tu objetivo:** Lograr un Accuracy en Test superior al **90%** sin que la diferencia entre entrenamiento y test sea mayor al 2%.



In [4]:
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import numpy as np

# Cargar datos
fashion_mnist = keras.datasets.fashion_mnist
(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()

# Normalización
train_images = train_images / 255.0
test_images = test_images / 255.0

# Añadir canal
train_images = train_images.reshape((train_images.shape[0], 28, 28, 1))
test_images = test_images.reshape((test_images.shape[0], 28, 28, 1))

# ============================================
# MODELO EQUILIBRADO (Sweet Spot)
# ============================================
mi_modelo = keras.Sequential([
    # Capa 1: 32 filtros (moderado)
    keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    keras.layers.MaxPooling2D((2, 2)),

    # Capa 2: 64 filtros (más capacidad)
    keras.layers.Conv2D(64, (3, 3), activation='relu'),
    keras.layers.MaxPooling2D((2, 2)),

    # Regularización para evitar overfitting
    keras.layers.Dropout(0.25),

    # Clasificador
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(10, activation='softmax')
])

# Compilar
mi_modelo.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("📊 Modelo equilibrado:")
mi_modelo.summary()

# Entrenar
print("\n🚀 Entrenando...")
history_mi_modelo = mi_modelo.fit(
    train_images, train_labels,
    epochs=15,
    validation_data=(test_images, test_labels),
    verbose=1
)

# Resultados finales
train_acc = history_mi_modelo.history['accuracy'][-1]
test_acc = history_mi_modelo.history['val_accuracy'][-1]
diferencia = abs(train_acc - test_acc)

print(f"\n{'='*50}")
print(f"📊 RESULTADOS FINALES:")
print(f"Precisión Entrenamiento: {train_acc:.4f} ({train_acc*100:.2f}%)")
print(f"Precisión Test: {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"Diferencia: {diferencia:.4f} ({diferencia*100:.2f}%)")
print(f"{'='*50}")

if test_acc >= 0.90:
    print("✅ ¡Test alcanzó o superó el 90%!")
else:
    print(f"⚠️ Test está en {test_acc*100:.1f}%. Faltan {(0.90 - test_acc)*100:.1f}%")

if diferencia <= 0.02:
    print("✅ ¡Diferencia menor al 2%! Buen equilibrio.")
else:
    print(f"⚠️ Diferencia del {diferencia*100:.1f}%. Hay overfitting.")

📊 Modelo equilibrado:


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 128)            │       204,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 225,034 (879.04 KB)

 Trainable params: 225,034 (879.04 KB)

 Non-trainable params: 0 (0.00 B)


🚀 Entrenando...
Epoch 1/15
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 37s 19ms/step - accuracy: 0.7929 - loss: 0.5728 - val_accuracy: 0.8641 - val_loss: 0.3777
Epoch 2/15
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 34s 18ms/step - accuracy: 0.8584 - loss: 0.3925 - val_accuracy: 0.8817 - val_loss: 0.3201
Epoch 3/15
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 40s 18ms/step - accuracy: 0.8750 - loss: 0.3433 - val_accuracy: 0.8929 - val_loss: 0.2917
Epoch 4/15
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 34s 18ms/step - accuracy: 0.8858 - loss: 0.3100 - val_accuracy: 0.8983 - val_loss: 0.2772
Epoch 5/15
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 34s 18ms/step - accuracy: 0.8933 - loss: 0.2923 - val_accuracy: 0.8900 - val_loss: 0.2956
Epoch 6/15
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 33s 18ms/step - accuracy: 0.9000 - loss: 0.2762 - val_accuracy: 0.9055 - val_loss: 0.2594
Epoch 7/15
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 34s 18ms/step - accuracy: 0.9029 - loss: 0.2639 - val_accuracy: 0.9062 - val_loss: 0.2625
Epoch 8/15
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 41s 18ms/step - a

Diseñé un modelo con capacidad moderada (32 y 64 filtros convolucionales) para capturar patrones complejos sin sobrecargar, y apliqué Dropout (25% y 50%) para regularizar y evitar el overfitting, logrando un equilibrio perfecto con 92% en entrenamiento y 91% en test.